# ETL: Online Retail II → витрины для Power BI


Ноутбук подготавливает данные Online Retail II для отчёта Power BI:
обзора бизнеса и интерактивного отбора SKU на дополнительную проверку.

ETL воспроизводит итоговые правила очистки и аналитические расчёты
исследовательского ноутбука. Повторный исследовательский анализ
и новые правила удаления наблюдений здесь не вводятся.

В транзакционной витрине сохраняются как продажи, так и товарные
возвраты. Сумма операции рассчитывается с исходным знаком количества:

**Amount = Quantity × Price**

Поэтому сумма Amount уже учитывает возвраты: отрицательные операции
уменьшают выручку, повторно вычитать их не нужно.

ABC, XYZ, RFM и адаптированная BCG рассчитываются по методике
исследовательского ноутбука. Выручку положительных продаж и выручку
с учётом возвратов храним как разные показатели.

Периоды анализа:
- 2009/10: 01.12.2009–30.11.2010;
- 2010/11: 01.12.2010–30.11.2011.

Этапы: загрузка → очистка → расчёт витрин → проверки → выгрузка.
Все ячейки выполняются последовательно сверху вниз.


## План ETL
1. Основные договорённости

 Очистку переносим из ноутбука, а не придумываем заново

ETL должен воспроизводить **финальный результат очистки**, включая уточнения, появившиеся позднее — например, исключение `CRUK`.

Исследовательские таблицы и графики каждый раз строить не нужно. В ETL переносим:
- принятые правила;
- необходимые преобразования;
- контрольные проверки.

 Разделяем состав данных и методику расчётов

В ноутбуке уже есть два уровня:

- `clean` — продажи **и** возвраты;
- положительные продажи — выборка для отдельных аналитических расчётов.

Поэтому наличие возвратов в `fact_sales` **не означает автоматический пересчёт всех классификаций по нетто-выручке**.

Предлагаю сохранить методику ноутбука:

| Показатель | Основа |
|---|---|
| Общая выручка с учётом возвратов | Все очищенные операции, сумма со знаком |
| Выручка положительных продаж | Только продажи |
| Сумма возвратов | Модуль отрицательных сумм, отдельно для контроля |
| ABC, XYZ, RFM, BCG | Выборки и формулы, использованные в ноутбуке |
| Денежное сальдо SKU | Продажи минус возвраты |

Так мы одновременно выполняем ваше требование к общей выручке и сохраняем воспроизводимость исследования. В выгрузках названия показателей должны явно различать **выручку продаж** и **выручку с учётом возвратов**.

 2. Правила очистки, которые необходимо повторить

| Объект | Решение из ноутбука |
|---|---|
| Полные дубликаты | Оставить первый экземпляр по восьми исходным бизнес-полям; `source_sheet` в ключ сравнения не включать |
| Совпадения по неполному ключу | Не удалять автоматически, если другие исходные поля отличаются |
| Признак отмены | Номер счёта после удаления крайних пробелов и приведения к верхнему регистру начинается с `C` |
| Допустимые операции | Положительное количество без `C` **или** отрицательное количество с `C` |
| Цена | Сохранить операции с `Price > 0` |
| Служебные позиции | Исключить установленный список кодов и префикс `GIFT_`, для продаж и возвратов одинаково |
| Пропуски покупателя | Сохранить в товарных операциях; не использовать для идентифицированного RFM |
| Необычные товарные коды | Не удалять только из-за формата кода |
| Экстремальные цены и количества | Не вводить нового автоматического отсечения |
| Названия товаров | Использовать принятую в ноутбуке обработку описаний и формирование `ProductName`; ключ товара — `StockCode` |
| Период | От `2009-12-01` включительно до `2011-12-01` исключительно |

Полный список исключаемых кодов с учётом позднего уточнения:

```text
M, DOT, POST, AMAZONFEE, C2, B,
ADJUST, ADJUST2, BANK CHARGES, D,
S, TEST001, TEST002, CRUK
```

**Важная особенность именно ваших данных:** исследование обнаружило отрицательные количества без `C`, но с нулевой ценой и признаками списаний/учётных корректировок. Поэтому сохраняем товарные возвраты, но не объявляем любую отрицательную строку исходника возвратом покупателя. Это соответствует итоговой очистке ноутбука.

 3. План реализации ETL

 Шаг 1. Зафиксировать параметры и правила

В одном месте задаём:

- путь к исходному файлу;
- папку выгрузки;
- границы двух периодов;
- восемь полей проверки дубликатов;
- список служебных кодов;
- действующие пороги классификаций и флагов.

**Результат:** небольшой блок настроек, без распределённых по коду констант.

 Шаг 2. Загрузить исходные данные

Используем существующий подход ноутбука: исходный Excel или уже подготовленный исходный Parquet.

Проверяем наличие обязательных колонок, типы и корректность дат. Не вводим скрытой дополнительной очистки: неожиданные проблемы должны попасть в проверку, а не исчезнуть незаметно.

**Результат:** `raw`.

 Шаг 3. Повторить итоговую очистку

Последовательно выполняем правила из таблицы выше:

1. Удаляем полные дубликаты.
2. Формируем признаки операций.
3. Исключаем нетоварные и недопустимые записи.
4. Ограничиваем период.
5. Рассчитываем `Amount` со знаком.
6. Присваиваем `Period`.

После каждого этапа записываем число строк до и после.

**Результат:** единая `clean`, содержащая продажи и возвраты.

 Шаг 4. Подготовить транзакционную витрину

Из **всей `clean`**, без фильтра на положительное количество, создаём `fact_sales`.

Сохраняем:

- идентификаторы операции, счёта, SKU и покупателя;
- дату и период;
- количество со знаком;
- цену;
- сумму со знаком;
- признак продажи/возврата;
- необходимые ключи для связей.

Возврат учитывается по дате его записи. Не пытаемся автоматически привязать его к исходному заказу и перенести в другой месяц.

**Результат:** `fact_sales`, где простая сумма `Amount` даёт выручку с учётом возвратов.

 Шаг 5. Воспроизвести аналитические витрины

Переносим существующие расчёты, не добавляя новые методы:

- `sku_period` — годовые метрики, ABC/XYZ, BCG, клиентские показатели и флаги;
- `sku_month` — месячные показатели с нулевыми месяцами;
- `customer_period` — RFM покупателей.

В месячной витрине разделяем:- проданное количество;
- возвращённое количество;
- выручку положительных продаж;
- сумму возвратов;
- выручку с учётом возвратов.

**XYZ продолжает использовать ряд положительно проданного количества**, как в исследовании, а не ряд количества после вычитания возвратов.

Отдельно проверяем SKU, у которых в периоде есть только возвраты: они не должны потеряться из общей выручки из-за отсутствия в таблице классификаций. Классы таким строкам искусственно не назначаем.

Шаг 6. Подготовить таблицы к Power BI

- Добавляем согласованные ключи.
- Создаём `dim_period` и `dim_date`.
- Приводим даты, числа, текст и флаги к экспортным типам.
- Сохраняем различие «Нет» и «Не проверено».
- Корректно привязываем BCG к периоду сравнения.
- Не создаём отдельные таблицы сценариев и кандидатов.

**Результат:** согласованный набор таблиц для загрузки.

Шаг 7. Проверить и выгрузить

Обязательные проверки:

- очистка воспроизводит итоговую выборку ноутбука;
- продажи и возвраты сохранены с правильными знаками;
- `Amount = Quantity × Price`;
- сумма всех операций равна продажам минус модуль возвратов;
- месячные суммы сходятся с транзакционными;
- годовые суммы сходятся на одинаковом охвате SKU;
- ключи витрин уникальны на заявленном уровне;
- возвраты не потеряны при соединениях;
- повторный запуск даёт тот же результат.

После успешных проверок выгружаем таблицы и короткий журнал очистки.

## Загрузка

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [2]:
# ─────────────────────────────────────────────────────
# ЗАГРУЗКА
# Одноразовая конвертация Excel → Parquet для скорости.
# Дальше читаем только parquet.
# ─────────────────────────────────────────────────────
RAW_PARQUET = Path("raw") / "online_retail_II.parquet"
if not RAW_PARQUET.exists():
    sheets = pd.read_excel(RAW_XLSX, sheet_name=None)

    raw = pd.concat(
        [s.assign(source_sheet=name) for name, s in sheets.items()],
        ignore_index=True,
    )

    # Типы — сразу под parquet
    raw["Invoice"]     = raw["Invoice"].astype(str)
    raw["StockCode"]   = raw["StockCode"].astype(str)
    raw["Description"] = raw["Description"].astype("string")
    raw["Quantity"]    = raw["Quantity"].astype("int32")
    raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"])
    raw["Price"]       = raw["Price"].astype("float64")
    raw["Customer ID"] = raw["Customer ID"].astype("Int32")
    raw["Country"]     = raw["Country"].astype("category")
    raw["source_sheet"] = raw["source_sheet"].astype("category")

    raw.to_parquet(RAW_PARQUET, index=False)
    print("Parquet создан:", RAW_PARQUET)
else:
    raw = pd.read_parquet(RAW_PARQUET)
    print("Parquet загружен:", RAW_PARQUET)

Parquet загружен: raw\online_retail_II.parquet


In [3]:
# ─────────────────────────────────────────────────────
# ВАЛИДАЦИЯ
# Проверяем, что данные пригодны для очистки.
# Ничего не меняем — только фиксируем факты.
# ─────────────────────────────────────────────────────

REQUIRED_COLS = {
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country",
    "source_sheet",
}

missing_cols = REQUIRED_COLS - set(raw.columns)
assert not missing_cols, f"Нет колонок: {missing_cols}"

print(f"Строк:           {len(raw):>10,}")
print(f"Колонок:         {raw.shape[1]}")
print(f"Период:          {raw['InvoiceDate'].min()} — {raw['InvoiceDate'].max()}")
print(f"Пропуск дат:     {raw['InvoiceDate'].isna().sum()}")

print("\nЛисты:")
print(raw["source_sheet"].value_counts())

print("\nПропуски по колонкам:")
missing = raw.isna().sum().to_frame("missing")
missing["pct"] = (100 * missing["missing"] / len(raw)).round(2)
display(missing.loc[missing["missing"] > 0])

print("\nДиапазон Quantity:", raw["Quantity"].min(), "—", raw["Quantity"].max())
print("Диапазон Price:   ", raw["Price"].min(),   "—", raw["Price"].max())

Строк:            1,067,371
Колонок:         9
Период:          2009-12-01 07:45:00 — 2011-12-09 12:50:00
Пропуск дат:     0

Листы:
source_sheet
Year 2010-2011    541910
Year 2009-2010    525461
Name: count, dtype: int64

Пропуски по колонкам:


,missing,pct
Description,4382,0.41
Customer ID,243007,22.77



Диапазон Quantity: -80995 — 80995
Диапазон Price:    -53594.36 — 38970.0


## Очистка

In [4]:
# ─────────────────────────────────────────────────────
# ОЧИСТКА
# Применяем контракт из шага 1.
# Результат: clean — продажи и возвраты, все товарные операции.
# ─────────────────────────────────────────────────────

from pathlib import Path
import numpy as np
import pandas as pd

# Параметры — локально, чтобы ячейка запускалась отдельно
P1_START = pd.Timestamp("2009-12-01")
P2_START = pd.Timestamp("2010-12-01")
P2_END   = pd.Timestamp("2011-12-01")

DUP_COLS = ["Invoice", "StockCode", "Description", "Quantity",
            "InvoiceDate", "Price", "Customer ID", "Country"]

EXCLUDE_CODES = {
    "M", "DOT", "POST", "AMAZONFEE", "C2", "B",
    "ADJUST", "ADJUST2", "BANK CHARGES", "D",
    "S", "TEST001", "TEST002", "CRUK",
}
EXCLUDE_PREFIXES = ("GIFT_",)

rows_before = len(raw)

# ─────────────────────────────────────────────────────
# 1. Дубли по 8 полям
# ─────────────────────────────────────────────────────
clean = raw.drop_duplicates(subset=DUP_COLS, keep="first").copy()
dup_removed = rows_before - len(clean)

# ─────────────────────────────────────────────────────
# 2. Признак отмены
# ─────────────────────────────────────────────────────
clean["IsCancel"] = (
    clean["Invoice"].astype("string").str.strip().str.upper()
    .str.startswith("C", na=False)
)

# ─────────────────────────────────────────────────────
# 3. Служебные позиции
# ─────────────────────────────────────────────────────
stock = clean["StockCode"].astype("string").str.strip().str.upper()
nonproduct = (
    stock.isin(EXCLUDE_CODES)
    | stock.str.startswith(EXCLUDE_PREFIXES, na=False)
)

# ─────────────────────────────────────────────────────
# 4. Допустимые операции
# ─────────────────────────────────────────────────────
# Продажа: без C, Quantity > 0
# Отмена:  с C, Quantity < 0
# Остальное (положительный C, отрицательный без C) — исключаем
valid_operation = (
    (~clean["IsCancel"] & clean["Quantity"].gt(0))
    | (clean["IsCancel"] & clean["Quantity"].lt(0))
)

# ─────────────────────────────────────────────────────
# 5. Цена > 0
# ─────────────────────────────────────────────────────
price_valid = clean["Price"].gt(0)

# ─────────────────────────────────────────────────────
# 6. Финальный фильтр
# ─────────────────────────────────────────────────────
keep = valid_operation & price_valid & ~nonproduct
clean = clean.loc[keep].copy()

# ─────────────────────────────────────────────────────
# 7. Сумма операции со знаком
# ─────────────────────────────────────────────────────
clean["Amount"] = clean["Quantity"] * clean["Price"]

# ─────────────────────────────────────────────────────
# 8. Период [P1_START, P2_END)
# ─────────────────────────────────────────────────────
clean = clean.loc[
    clean["InvoiceDate"].ge(P1_START)
    & clean["InvoiceDate"].lt(P2_END)
].copy()

clean["Period"] = pd.cut(
    clean["InvoiceDate"],
    bins=pd.to_datetime([P1_START, P2_START, P2_END]),
    labels=["2009/10", "2010/11"],
    right=False,
)

clean = clean.reset_index(drop=True)

In [5]:
# ─────────────────────────────────────────────────────
# КОНТРОЛЬ ПРАВИЛ
# ─────────────────────────────────────────────────────
assert clean["Price"].gt(0).all(), "Остались цены ≤ 0"
assert clean.loc[~clean["IsCancel"], "Quantity"].gt(0).all(), "Продажа с Quantity ≤ 0"
assert clean.loc[clean["IsCancel"], "Quantity"].lt(0).all(), "Отмена с Quantity ≥ 0"
assert clean["Period"].notna().all(), "Есть строки без периода"
assert (clean["Amount"] == clean["Quantity"] * clean["Price"]).all()

# ─────────────────────────────────────────────────────
# ЖУРНАЛ
# ─────────────────────────────────────────────────────
print("ОЧИСТКА — ЖУРНАЛ")
print(f"  Строк до очистки:              {rows_before:>10,}")
print(f"  Удалено полных дублей:         {dup_removed:>10,}")
print(f"  Осталось в clean:              {len(clean):>10,}")
print(f"    Продажи:                     {(~clean['IsCancel']).sum():>10,}")
print(f"    Отмены / возвраты:           {clean['IsCancel'].sum():>10,}")
print()
print(f"  Продажи с Customer ID:         "
      f"{(~clean['IsCancel'] & clean['Customer ID'].notna()).sum():>10,}")
print(f"  Продажи без Customer ID:       "
      f"{(~clean['IsCancel'] & clean['Customer ID'].isna()).sum():>10,}")
print()

print("ПО ПЕРИОДАМ")
display(
    clean.groupby("Period", observed=True)
    .agg(
        rows=("Invoice", "size"),
        sales=("IsCancel", lambda s: (~s).sum()),
        returns=("IsCancel", "sum"),
        amount=("Amount", "sum"),
    )
    .round(2)
)

print("\nДЕНЕЖНЫЙ ИТОГ")
print(f"  Продажи (положительный Amount):  "
      f"{clean.loc[clean['Amount'] > 0, 'Amount'].sum():>14,.2f} GBP")
print(f"  Возвраты (модуль):               "
      f"{-clean.loc[clean['Amount'] < 0, 'Amount'].sum():>14,.2f} GBP")
print(f"  Сальдо = SUM(Amount):            "
      f"{clean['Amount'].sum():>14,.2f} GBP")

ОЧИСТКА — ЖУРНАЛ
  Строк до очистки:               1,067,371
  Удалено полных дублей:             34,335
  Осталось в clean:                 996,172
    Продажи:                        978,604
    Отмены / возвраты:               17,568

  Продажи с Customer ID:            759,632
  Продажи без Customer ID:          218,972

ПО ПЕРИОДАМ


,rows,sales,returns,amount
Period,,,,
2009/10,490064,480817,9247,"9,156,080.90"
2010/11,506108,497787,8321,"9,330,918.60"



ДЕНЕЖНЫЙ ИТОГ
  Продажи (положительный Amount):   19,029,370.72 GBP
  Возвраты (модуль):                   542,371.21 GBP
  Сальдо = SUM(Amount):             18,486,999.51 GBP


## Транзакцтонная витрина

In [6]:
# ─────────────────────────────────────────────────────
# FACT_SALES
# Из всей clean — продажи и возвраты.
# Amount со знаком: SUM(Amount) = выручка с учётом возвратов.
# ─────────────────────────────────────────────────────

fact_sales = clean[[
    "Invoice", "StockCode", "Customer ID",
    "InvoiceDate", "Quantity", "Price", "Amount",
    "IsCancel", "Period", "Country",
]].copy()

# Переименование под Power BI
fact_sales = fact_sales.rename(columns={
    "Invoice":     "InvoiceNo",
    "Customer ID": "CustomerID",
    "InvoiceDate": "Date",
})

# Ключ строки для уникальности
fact_sales = fact_sales.reset_index(drop=True)
fact_sales["RowID"] = fact_sales.index + 1

# Типы для экспорта
fact_sales["Date"]       = pd.to_datetime(fact_sales["Date"])
fact_sales["Quantity"]   = fact_sales["Quantity"].astype("int32")
fact_sales["Price"]      = fact_sales["Price"].astype("float64")
fact_sales["Amount"]     = fact_sales["Amount"].astype("float64")
fact_sales["IsCancel"]   = fact_sales["IsCancel"].astype(bool)
fact_sales["Period"]     = fact_sales["Period"].astype("category")
fact_sales["Country"]    = fact_sales["Country"].astype("category")

# Порядок колонок
fact_sales = fact_sales[[
    "RowID", "InvoiceNo", "StockCode", "CustomerID",
    "Date", "Quantity", "Price", "Amount",
    "IsCancel", "Period", "Country",
]]

print(f"fact_sales: {len(fact_sales):,} строк")
print(f"Колонки: {fact_sales.columns.tolist()}")
display(fact_sales.head())

fact_sales: 996,172 строк
Колонки: ['RowID', 'InvoiceNo', 'StockCode', 'CustomerID', 'Date', 'Quantity', 'Price', 'Amount', 'IsCancel', 'Period', 'Country']


,RowID,InvoiceNo,StockCode,CustomerID,Date,Quantity,Price,Amount,IsCancel,Period,Country
0,1,489434,85048,13085,2009-12-01 07:45:00,12,6.95,83.40,False,2009/10,United Kingdom
1,2,489434,79323P,13085,2009-12-01 07:45:00,12,6.75,81.00,False,2009/10,United Kingdom
2,3,489434,79323W,13085,2009-12-01 07:45:00,12,6.75,81.00,False,2009/10,United Kingdom
3,4,489434,22041,13085,2009-12-01 07:45:00,48,2.10,100.80,False,2009/10,United Kingdom
4,5,489434,21232,13085,2009-12-01 07:45:00,24,1.25,30.00,False,2009/10,United Kingdom


In [7]:
# ─────────────────────────────────────────────────────
# КОНТРОЛЬ
# ─────────────────────────────────────────────────────
assert len(fact_sales) == len(clean), "Потеряны строки при сборке"
assert fact_sales["RowID"].is_unique, "RowID не уникален"
assert (fact_sales["Amount"] == fact_sales["Quantity"] * fact_sales["Price"]).all()
assert fact_sales.loc[~fact_sales["IsCancel"], "Amount"].gt(0).all(), "Продажа с Amount ≤ 0"
assert fact_sales.loc[fact_sales["IsCancel"], "Amount"].lt(0).all(), "Отмена с Amount ≥ 0"

print("Проверки fact_sales: OK")

# Сверка с clean
print(f"\nСальдо clean:       {clean['Amount'].sum():>14,.2f}")
print(f"Сальдо fact_sales:  {fact_sales['Amount'].sum():>14,.2f}")
print(f"Разница:            {clean['Amount'].sum() - fact_sales['Amount'].sum():>14,.2f}")

# Разбивка по периодам
print("\nПо периодам:")
display(
    fact_sales.groupby("Period", observed=True)
    .agg(
        rows=("RowID", "size"),
        sales=("IsCancel", lambda s: (~s).sum()),
        returns=("IsCancel", "sum"),
        amount=("Amount", "sum"),
    )
    .round(2)
)

Проверки fact_sales: OK

Сальдо clean:        18,486,999.51
Сальдо fact_sales:   18,486,999.51
Разница:                      0.00

По периодам:


,rows,sales,returns,amount
Period,,,,
2009/10,490064,480817,9247,"9,156,080.90"
2010/11,506108,497787,8321,"9,330,918.60"


получили

**`fact_sales`** — транзакционная витрина для страницы «Обзор бизнеса»:

| Колонка | Смысл |
|---|---|
| `RowID` | Уникальный ключ строки |
| `InvoiceNo` | Номер счёта |
| `StockCode` | Код товара |
| `CustomerID` | Покупатель (может быть пустым) |
| `Date` | Дата операции |
| `Quantity` | Количество со знаком |
| `Price` | Цена за единицу |
| `Amount` | Сумма со знаком = Quantity × Price |
| `IsCancel` | `True` для отмен |
| `Period` | `2009/10` или `2010/11` |
| `Country` | Страна покупателя |

**Ключевые свойства:**
- Содержит **продажи и возвраты** — и положительные, и отрицательные строки.
- `SUM(Amount)` = выручка с учётом возвратов.
- `SUM(Amount WHERE Amount > 0)` = выручка продаж.
- `SUM(ABS(Amount) WHERE Amount < 0)` = сумма возвратов.
- **Повторно вычитать возвраты нельзя** — они уже входят со знаком минус.

## Аналитические витрины

In [8]:
# ─────────────────────────────────────────────────────
# SKU_PERIOD. ABC + базовые метрики
# Основа: положительные продажи (как в ноутбуке).
# Возвраты учитываем отдельно — в сальдо.
# ─────────────────────────────────────────────────────

sales = clean.loc[~clean["IsCancel"]].copy()

sku = (
    sales.groupby(["Period", "StockCode"], observed=True)
    .agg(
        ProductName=("Description", lambda s: s.mode().iloc[0]
                     if not s.mode().empty else "Без описания"),
        revenue=("Amount", "sum"),
        units=("Quantity", "sum"),
        orders=("Invoice", "nunique"),
    )
    .reset_index()
)

# ABC по выручке, отдельно в каждом периоде
sku = sku.sort_values(
    ["Period", "revenue", "StockCode"],
    ascending=[True, False, True]
).reset_index(drop=True)

sku["revenue_share_pct"] = (
    100 * sku["revenue"]
    / sku.groupby("Period")["revenue"].transform("sum")
)
sku["cumulative_share_pct"] = (
    sku.groupby("Period")["revenue_share_pct"].cumsum()
)

ABC_CUTS = {"A": 80, "B": 95}
sku["ABC"] = np.select(
    [sku["cumulative_share_pct"] <= ABC_CUTS["A"],
     sku["cumulative_share_pct"] <= ABC_CUTS["B"]],
    ["A", "B"], default="C"
)

print(f"sku_period (ABC): {len(sku):,} строк")
display(sku.head())

sku_period (ABC): 8,126 строк


C:\Users\urize\AppData\Local\Temp\ipykernel_13460\4078574100.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  / sku.groupby("Period")["revenue"].transform("sum")
C:\Users\urize\AppData\Local\Temp\ipykernel_13460\4078574100.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sku.groupby("Period")["revenue_share_pct"].cumsum()


,Period,StockCode,ProductName,revenue,units,orders,revenue_share_pct,cumulative_share_pct,ABC
0,2009/10,22423,REGENCY CAKESTAND 3 TIER,"156,433.78",12627,1930,1.66,1.66,A
1,2009/10,85123A,WHITE HANGING HEART T-LIGHT HOLDER,"153,261.96",56562,3167,1.63,3.30,A
2,2009/10,85099B,JUMBO BAG RED RETROSPOT,"86,409.53",48386,1900,0.92,4.22,A
3,2009/10,84879,ASSORTED COLOUR BIRD ORNAMENT,"70,396.87",43720,1352,0.75,4.96,A
4,2009/10,22086,PAPER CHAIN KIT 50'S CHRISTMAS,"52,884.70",15755,858,0.56,5.53,A


In [9]:
# ─────────────────────────────────────────────────────
# XYZ. Полный ряд SKU × месяц, нули обязательны.
# Основа: положительно проданное количество.
# ─────────────────────────────────────────────────────

PERIOD_MONTHS = {
    "2009/10": pd.period_range("2009-12", "2010-11", freq="M"),
    "2010/11": pd.period_range("2010-12", "2011-11", freq="M"),
}

parts = []
for period, month_range in PERIOD_MONTHS.items():
    sales_p = sales.loc[sales["Period"].eq(period)]
    skus_p = sku.loc[sku["Period"].eq(period), "StockCode"]

    grid = pd.MultiIndex.from_product(
        [skus_p, month_range],
        names=["StockCode", "Month"]
    )

    monthly = (
        sales_p
        .assign(Month=sales_p["InvoiceDate"].dt.to_period("M"))
        .groupby(["StockCode", "Month"], observed=True)["Quantity"]
        .sum()
        .reindex(grid, fill_value=0)
        .rename("units")
        .reset_index()
        .assign(Period=period)
    )
    parts.append(monthly)

sku_month_sales = pd.concat(parts, ignore_index=True)

# Контроль: ровно 12 месяцев на SKU
assert (sku_month_sales
        .groupby(["Period", "StockCode"])["Month"]
        .nunique().eq(12).all()), "Не у всех SKU 12 месяцев"

# XYZ-метрики
xyz = (
    sku_month_sales
    .groupby(["Period", "StockCode"], observed=True)
    .agg(
        active_months=("units", lambda x: x.gt(0).sum()),
        mean_qty=("units", "mean"),
        std_qty=("units", lambda x: x.std(ddof=0)),
    )
    .reset_index()
)
xyz["CV"] = xyz["std_qty"] / xyz["mean_qty"]

XYZ_CUTS = {"X": 0.75, "Y": 1.5}
xyz["XYZ"] = np.select(
    [xyz["CV"] <= XYZ_CUTS["X"], xyz["CV"] <= XYZ_CUTS["Y"]],
    ["X", "Y"], default="Z"
)

sku = sku.merge(
    xyz[["Period", "StockCode", "active_months",
         "mean_qty", "std_qty", "CV", "XYZ"]],
    on=["Period", "StockCode"], how="left",
    validate="one_to_one"
)
sku["ABC_XYZ"] = sku["ABC"] + sku["XYZ"]

print(f"sku_period (ABC+XYZ): {len(sku):,} строк")

sku_period (ABC+XYZ): 8,126 строк


In [10]:
# ─────────────────────────────────────────────────────
# SKU_MONTH. Продажи + возвраты, раздельно.
# Для графиков и для Power BI.
# ─────────────────────────────────────────────────────

def build_sku_month_side(df, sign, suffix):
    """Собрать месячную агрегацию для продаж или возвратов."""
    subset = df.loc[df["IsCancel"].eq(False)] if sign > 0 else df.loc[df["IsCancel"]]

    parts = []
    for period, month_range in PERIOD_MONTHS.items():
        sub = subset.loc[subset["Period"].eq(period)]
        skus_p = sku.loc[sku["Period"].eq(period), "StockCode"]

        grid = pd.MultiIndex.from_product(
            [skus_p, month_range],
            names=["StockCode", "Month"]
        )

        agg = (
            sub
            .assign(Month=sub["InvoiceDate"].dt.to_period("M"))
            .groupby(["StockCode", "Month"], observed=True)
            .agg(qty=("Quantity", "sum"), amount=("Amount", "sum"))
            .reindex(grid, fill_value=0)
            .reset_index()
            .assign(Period=period)
        )
        parts.append(agg)
    return pd.concat(parts, ignore_index=True)


sku_month = (
    build_sku_month_side(clean, +1, "sales")
    .rename(columns={"qty": "qty_sold", "amount": "revenue_sold"})
    .merge(
        build_sku_month_side(clean, -1, "returns")
        .rename(columns={"qty": "qty_returned", "amount": "amount_returned"}),
        on=["Period", "StockCode", "Month"], how="outer",
        validate="one_to_one"
    )
)

sku_month = sku_month.fillna(0)
sku_month["qty_returned"]    = sku_month["qty_returned"].abs()
sku_month["amount_returned"] = sku_month["amount_returned"].abs()
sku_month["revenue_net"]     = (
    sku_month["revenue_sold"] - sku_month["amount_returned"]
)
sku_month["Month"] = sku_month["Month"].dt.to_timestamp()

print(f"sku_month: {len(sku_month):,} строк")
display(sku_month.head())

sku_month: 97,512 строк


,StockCode,Month,qty_sold,revenue_sold,Period,qty_returned,amount_returned,revenue_net
0,10002,2009-12-01,215,185.30,2009/10,0,0.00,185.30
1,10002,2010-01-01,291,248.97,2009/10,0,0.00,248.97
2,10002,2010-02-01,257,220.07,2009/10,0,0.00,220.07
3,10002,2010-03-01,641,493.09,2009/10,0,0.00,493.09
4,10002,2010-04-01,1132,843.73,2009/10,200,144.00,699.73


In [11]:
# ─────────────────────────────────────────────────────
# BCG. Рост выручки 2010/11 к 2009/10.
# Только SKU с продажами в оба периода.
# ─────────────────────────────────────────────────────

rev_pivot = (
    sku.pivot(index="StockCode", columns="Period", values="revenue")
    .rename(columns={"2009/10": "revenue_p1", "2010/11": "revenue_p2"})
    .fillna(0)
)

rev_pivot["growth_pct"] = np.where(
    rev_pivot["revenue_p1"] > 0,
    (rev_pivot["revenue_p2"] / rev_pivot["revenue_p1"] - 1) * 100,
    np.nan
)
rev_pivot["revenue_change"] = (
    rev_pivot["revenue_p2"] - rev_pivot["revenue_p1"]
)

abc_p2 = (
    sku.loc[sku["Period"].eq("2010/11"), ["StockCode", "ABC"]]
    .rename(columns={"ABC": "ABC_p2"})
    .set_index("StockCode")
)

bcg = rev_pivot.join(abc_p2, how="left")
high_share = bcg["ABC_p2"].eq("A")
change = bcg["revenue_change"]

bcg["BCG_adapted"] = np.select(
    [high_share & change.gt(0),
     high_share & change.lt(0),
     ~high_share & change.gt(0),
     ~high_share & change.lt(0)],
    ["A — рост", "A — снижение", "B/C — рост", "B/C — снижение"],
    default="Без изменения"
)
bcg["growth_pct"] = bcg["growth_pct"].fillna(0)

sku["BCG_adapted"] = sku["StockCode"].map(bcg["BCG_adapted"])

print("BCG присвоен")
print(bcg["BCG_adapted"].value_counts())

BCG присвоен
BCG_adapted
B/C — снижение    2825
B/C — рост        1228
A — рост           512
A — снижение       320
Без изменения       10
Name: count, dtype: int64


In [12]:
# ─────────────────────────────────────────────────────
# ДОПОЛНИТЕЛЬНЫЕ ПРИЗНАКИ SKU
# История, сезонные пики, сальдо, Champions/Loyal.
# ─────────────────────────────────────────────────────

# 1) История товара
sales_only = clean.loc[~clean["IsCancel"]]
first_sale = sales_only.groupby("StockCode", observed=True)["InvoiceDate"].min()

assess_dates = {
    "2009/10": pd.Timestamp("2010-12-01"),
    "2010/11": pd.Timestamp("2011-12-01"),
}
sku["first_observed_sale"] = sku["StockCode"].map(first_sale)
eval_date = sku["Period"].astype("string").map(assess_dates).pipe(pd.to_datetime)
sku["observed_history_days"] = (
    eval_date - sku["first_observed_sale"].dt.normalize()
).dt.days
sku["short_history_flag"] = sku["observed_history_days"].le(90)


# 2) Сезонные пики — ровно как в ноутбуке «Навигатор»
#    Условия: топ-3 месяца дают ≥ 50% в КАЖДОМ году
#             И минимум 2 пиковых месяца совпадают между годами.

# 2.1 Полные месячные ряды: SKU × календарный месяц × период
monthly_by_period = (
    sku_month
    .assign(calendar_month=sku_month["Month"].dt.month)
    .pivot_table(
        index=["Period", "StockCode"],
        columns="calendar_month",
        values="qty_sold",
        aggfunc="sum",
        observed=True,
        fill_value=0,
    )
    .reindex(columns=range(1, 13), fill_value=0)
    .fillna(0)
)

# 2.2 Разделяем по периодам, берём только сопоставимые SKU
p1 = monthly_by_period.xs("2009/10")
p2 = monthly_by_period.xs("2010/11")
common_sku = p1.index.intersection(p2.index)

# 2.3 Топ-3 месяца внутри каждого периода отдельно
def peaks(row):
    """Множество месяцев с тремя крупнейшими объёмами продаж."""
    return set(row[row.gt(0)].nlargest(3).index)

# 2.4 Доля топ-3 месяцев в общем объёме — отдельно для каждого года
share1 = p1.loc[common_sku].apply(
    lambda x: x.nlargest(3).sum() / x.sum() if x.sum() > 0 else 0,
    axis=1,
)
share2 = p2.loc[common_sku].apply(
    lambda x: x.nlargest(3).sum() / x.sum() if x.sum() > 0 else 0,
    axis=1,
)

# 2.5 Реальное пересечение пиковых месяцев между годами
overlap = pd.Series(
    [
        len(a & b)
        for a, b in zip(
            p1.loc[common_sku].apply(peaks, axis=1),
            p2.loc[common_sku].apply(peaks, axis=1),
        )
    ],
    index=common_sku,
)

# 2.6 Итоговый флаг: три условия одновременно
seasonal_mask = (
    share1.ge(0.5)
    & share2.ge(0.5)
    & overlap.ge(2)
)

# 2.7 Переносим флаг в витрину только для второго периода
sku["repeated_peak_flag"] = (
    sku["StockCode"]
    .map(seasonal_mask)
    .where(sku["Period"].eq("2010/11"))
    .astype("boolean")
)

# Явный признак, что сезонность проверена (для Power BI —
# отличить «не сезонный» от «не проверялся»)
sku["seasonality_checked"] = sku["repeated_peak_flag"].notna()


# 3) Сальдо = выручка продаж − модуль возвратов
negative_returns = (
    clean.loc[clean["IsCancel"]]
    .groupby(["Period", "StockCode"], observed=True)["Amount"]
    .sum().abs().rename("negative_amount")
)
sku = sku.merge(
    negative_returns, on=["Period", "StockCode"], how="left",
    validate="one_to_one"
)
sku["negative_amount"] = sku["negative_amount"].fillna(0)
sku["sales_balance"] = sku["revenue"] - sku["negative_amount"]
sku["negative_to_sales_pct"] = (
    100 * sku["negative_amount"] / sku["revenue"]
)

print("Признаки истории, сезонности, сальдо добавлены")
print(f"short_history_flag:  {sku['short_history_flag'].sum()} SKU")
print(f"repeated_peak_flag:  {sku['repeated_peak_flag'].fillna(False).sum()} SKU")

# Контроль: сверка с ноутбуком
seasonal_p2 = sku.loc[
    sku["Period"].eq("2010/11"),
    "repeated_peak_flag",
]
print("\nРаспределение repeated_peak_flag в 2010/11:")
print(seasonal_p2.value_counts(dropna=False))

Признаки истории, сезонности, сальдо добавлены
short_history_flag:  540 SKU
repeated_peak_flag:  531 SKU

Распределение repeated_peak_flag в 2010/11:
repeated_peak_flag
False    2700
<NA>      674
True      531
Name: count, dtype: Int64


In [13]:
# ─────────────────────────────────────────────────────
# RFM + КЛИЕНТСКИЕ МЕТРИКИ НА SKU
# ─────────────────────────────────────────────────────

customer_sales = clean.loc[
    ~clean["IsCancel"] & clean["Customer ID"].notna()
].copy()

# ── RFM на покупателя × период ──
parts = []
for period, date in assess_dates.items():
    d = customer_sales.loc[customer_sales["Period"].eq(period)]
    t = (
        d.groupby("Customer ID", observed=True)
        .agg(last_purchase=("InvoiceDate", "max"),
             Frequency=("Invoice", "nunique"),
             Monetary=("Amount", "sum"))
        .reset_index()
    )
    t["Recency"] = (date - t["last_purchase"].dt.normalize()).dt.days
    t["Period"] = period
    parts.append(t)

rfm = pd.concat(parts, ignore_index=True)

for period, g in rfm.groupby("Period", observed=True):
    idx = g.index
    rfm.loc[idx, "R_score"] = pd.qcut(g["Recency"].rank(method="first"),
                                      3, labels=[3, 2, 1]).astype(int)
    rfm.loc[idx, "F_score"] = pd.qcut(g["Frequency"].rank(method="first"),
                                      3, labels=[1, 2, 3]).astype(int)
    rfm.loc[idx, "M_score"] = pd.qcut(g["Monetary"].rank(method="first"),
                                      3, labels=[1, 2, 3]).astype(int)

r, f, m = rfm["R_score"], rfm["F_score"], rfm["M_score"]
rfm["RFM_segment"] = np.select(
    [r.eq(3) & f.eq(3) & m.eq(3),
     r.ge(2) & f.ge(2),
     r.eq(3) & f.eq(1),
     r.eq(1) & f.ge(2),
     r.eq(1)],
    ["Champions", "Loyal", "Recent single", "At Risk", "Inactive"],
    default="Regular"
)

# ── Клиентские метрики SKU × период ──
key_segments = ["Champions", "Loyal"]
customer_sku = (
    customer_sales[["Period", "StockCode", "Customer ID", "Amount"]]
    .merge(rfm[["Period", "Customer ID", "RFM_segment"]],
           on=["Period", "Customer ID"], how="left",
           validate="many_to_one")
)
key_mask = customer_sku["RFM_segment"].isin(key_segments)

sku_client = (
    customer_sku
    .assign(
        known_cust=customer_sku["Customer ID"],
        key_cust=customer_sku["Customer ID"].where(key_mask),
        key_rev=customer_sku["Amount"].where(key_mask, 0),
    )
    .groupby(["Period", "StockCode"], observed=True)
    .agg(
        known_customers=("known_cust", "nunique"),
        key_customers=("key_cust", "nunique"),
        key_customer_revenue=("key_rev", "sum"),
    )
    .reset_index()
)

sku = sku.merge(sku_client, on=["Period", "StockCode"],
                how="left", validate="one_to_one")
sku[["known_customers", "key_customers"]] = (
    sku[["known_customers", "key_customers"]].fillna(0).astype("int64")
)
sku["key_customer_revenue"] = sku["key_customer_revenue"].fillna(0)
sku["key_customer_revenue_share_pct"] = (
    100 * sku["key_customer_revenue"] / sku["revenue"]
)

print(f"sku_period итог: {len(sku):,} строк, {sku.shape[1]} колонок")
print(f"rfm: {len(rfm):,} строк")

sku_period итог: 8,126 строк, 28 колонок
rfm: 8,532 строк


Что получили

| Витрина | Гранулярность | Что содержит |
|---|---|---|
| `sku_period` | SKU × период | ABC, XYZ, ABC_XYZ, BCG, история, сезонность, сальдо, RFM-метрики |
| `sku_month` | SKU × месяц | проданное количество, возвращённое, выручка продаж, сумма возвратов, нетто |
| `rfm` | покупатель × период | Recency, Frequency, Monetary, баллы, сегмент |

**Ключевые решения:**
- ABC, XYZ — только по положительным продажам, как в ноутбуке.
- Возвраты живут в сальдо `sales_balance` — повторно не вычитаются.
- Сезонный флаг — только для второго периода, у кого есть продажи в оба года.
- RFM-метрики переносятся на SKU через Champions/Loyal.


## Подготовка к Power BI и выгрузка

### Справочники

In [14]:
# ─────────────────────────────────────────────────────
# СПРАВОЧНИКИ ДЛЯ POWER BI
# ─────────────────────────────────────────────────────

from pathlib import Path
PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# ── DimProduct: один SKU = одна строка ──
dim_product = (
    sku[["StockCode", "ProductName"]]
    .drop_duplicates("StockCode")
    .reset_index(drop=True)
)
dim_product["StockCode"] = dim_product["StockCode"].astype("string")
dim_product["ProductName"] = dim_product["ProductName"].astype("string")

# ── DimDate: календарь на весь период данных ──
dates = pd.date_range(
    clean["InvoiceDate"].min().normalize(),
    clean["InvoiceDate"].max().normalize(),
    freq="D",
)
dim_date = pd.DataFrame({"Date": dates})
dim_date["Year"]        = dim_date["Date"].dt.year
dim_date["Month"]       = dim_date["Date"].dt.month
dim_date["MonthName"]   = dim_date["Date"].dt.strftime("%b")
dim_date["Period"]      = pd.cut(
    dim_date["Date"],
    bins=pd.to_datetime(["2009-12-01", "2010-12-01", "2011-12-01"]),
    labels=["2009/10", "2010/11"],
    right=False,
)

# ── DimCustomer: уникальные покупатели ──
dim_customer = (
    clean.dropna(subset=["Customer ID"])
    .groupby("Customer ID", observed=True)
    .agg(Country=("Country", lambda s: s.mode().iloc[0]))
    .reset_index()
    .rename(columns={"Customer ID": "CustomerID"})
)
dim_customer["CustomerID"] = dim_customer["CustomerID"].astype("int64")
dim_customer["Country"]    = dim_customer["Country"].astype("category")

# ── DimPeriod: два периода с датами ──
dim_period = pd.DataFrame({
    "Period":    ["2009/10", "2010/11"],
    "DateStart": pd.to_datetime(["2009-12-01", "2010-12-01"]),
    "DateEnd":   pd.to_datetime(["2010-11-30", "2011-11-30"]),
})

print("DimProduct: ", len(dim_product))
print("DimDate:    ", len(dim_date))
print("DimCustomer:", len(dim_customer))
print("DimPeriod:  ", len(dim_period))

DimProduct:  4895
DimDate:     730
DimCustomer: 5847
DimPeriod:   2


## Приведение типов витрин

In [15]:
# ─────────────────────────────────────────────────────
# ТИПЫ ДЛЯ ЭКСПОРТА
# Унификация для быстрой загрузки в Power BI.
# ─────────────────────────────────────────────────────

def to_parquet_ready(df):
    """Приведение типов: категориальные где уместно, даты в datetime."""
    df = df.copy()

    # Строковые
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]):
            df[col] = df[col].astype("string")

    # Категориальные — включая флаги, чтобы Power BI видел их
    # как категорию с тремя значениями: True / False / (blank)
    for col in [
        "Period", "ABC", "XYZ", "ABC_XYZ", "BCG_adapted",
        "RFM_segment", "Country",
        "repeated_peak_flag", "short_history_flag", "seasonality_checked",
    ]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    # Даты — errors="coerce" вместо deprecated errors="ignore"
    for col in [
        "Date", "Month", "last_purchase",
        "first_observed_sale", "DateStart", "DateEnd",
    ]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


# --- sku_period ---
sku_period = sku.copy()
# Оставляем только рабочие поля для Power BI
sku_period_cols = [
    "Period", "StockCode", "ProductName",
    "revenue", "units", "orders", "revenue_share_pct",
    "cumulative_share_pct", "ABC",
    "active_months", "mean_qty", "std_qty", "CV", "XYZ", "ABC_XYZ",
    "first_observed_sale", "observed_history_days", "short_history_flag",
    "repeated_peak_flag",
    "negative_amount", "sales_balance", "negative_to_sales_pct",
    "known_customers", "key_customers",
    "key_customer_revenue", "key_customer_revenue_share_pct",
    "BCG_adapted",
]
sku_period = to_parquet_ready(
    sku_period[[c for c in sku_period_cols if c in sku_period.columns]]
)

# --- sku_month ---
sku_month = to_parquet_ready(sku_month)

# --- fact_sales ---
fact_sales = to_parquet_ready(fact_sales)

# --- rfm ---
rfm_export = to_parquet_ready(
    rfm.rename(columns={"Customer ID": "CustomerID"})
)

# --- справочники ---
dim_product  = to_parquet_ready(dim_product)
dim_date     = to_parquet_ready(dim_date)
dim_customer = to_parquet_ready(dim_customer)
dim_period   = to_parquet_ready(dim_period)

print("Типы приведены")

Типы приведены


## Финальные проверки

In [16]:
# ─────────────────────────────────────────────────────
# ПРОВЕРКИ ПЕРЕД ВЫГРУЗКОЙ
# ─────────────────────────────────────────────────────

# 1) Уникальность ключей
assert sku_period.duplicated(["Period", "StockCode"]).sum() == 0
assert sku_month.duplicated(["Period", "StockCode", "Month"]).sum() == 0
assert rfm_export.duplicated(["Period", "CustomerID"]).sum() == 0
assert dim_product["StockCode"].is_unique
assert dim_customer["CustomerID"].is_unique

# 2) Сумма Amount в fact_sales = сумме Amount в clean
assert abs(fact_sales["Amount"].sum() - clean["Amount"].sum()) < 1e-6

# 3) Выручка продаж в sku_period совпадает с фактом
rev_sku = sku_period.groupby("Period", observed=True)["revenue"].sum()
rev_fact = (
    clean.loc[~clean["IsCancel"]]
    .groupby("Period", observed=True)["Amount"].sum()
)
assert np.allclose(rev_sku.values, rev_fact.values), "Расхождение выручки"

# 4) Нет пропусков в ключевых полях
assert sku_period[["StockCode", "Period", "ABC", "XYZ"]].notna().all().all()

# 5) Сезонный флаг только для второго периода
assert sku_period.loc[
    sku_period["Period"].eq("2009/10"), "repeated_peak_flag"
].isna().all()

print("Проверки перед выгрузкой: OK")
print(f"sku_period:  {len(sku_period):,}")
print(f"sku_month:   {len(sku_month):,}")
print(f"fact_sales:  {len(fact_sales):,}")
print(f"rfm:         {len(rfm_export):,}")

Проверки перед выгрузкой: OK
sku_period:  8,126
sku_month:   97,512
fact_sales:  996,172
rfm:         8,532


## Выгрузка

In [17]:
# ─────────────────────────────────────────────────────
# ЭКСПОРТ В data/processed/
# Одна команда сохраняет все таблицы для Power BI.
# ─────────────────────────────────────────────────────

tables = {
    "DimProduct":   dim_product,
    "DimDate":      dim_date,
    "DimCustomer":  dim_customer,
    "DimPeriod":    dim_period,
    "FactSales":    fact_sales,
    "MartSkuPeriod": sku_period,
    "MartSkuMonth":  sku_month,
    "MartRfm":       rfm_export,
}

for name, df in tables.items():
    path = PROCESSED / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"{name:<18} {len(df):>8,} строк  →  {path}")

DimProduct            4,895 строк  →  data\processed\DimProduct.parquet
DimDate                 730 строк  →  data\processed\DimDate.parquet
DimCustomer           5,847 строк  →  data\processed\DimCustomer.parquet
DimPeriod                 2 строк  →  data\processed\DimPeriod.parquet
FactSales           996,172 строк  →  data\processed\FactSales.parquet
MartSkuPeriod         8,126 строк  →  data\processed\MartSkuPeriod.parquet
MartSkuMonth         97,512 строк  →  data\processed\MartSkuMonth.parquet
MartRfm               8,532 строк  →  data\processed\MartRfm.parquet


## Финальный журнал

In [18]:
# ─────────────────────────────────────────────────────
# ЖУРНАЛ ETL
# Сохраняем отчёт о результатах прогона.
# ─────────────────────────────────────────────────────

import json

summary = {
    "raw_rows":           int(len(raw)),
    "clean_rows":         int(len(clean)),
    "sales_rows":         int((~clean["IsCancel"]).sum()),
    "return_rows":        int(clean["IsCancel"].sum()),
    "revenue_sales_gbp":  round(float(clean.loc[~clean["IsCancel"], "Amount"].sum()), 2),
    "returns_gbp":        round(float(-clean.loc[clean["IsCancel"], "Amount"].sum()), 2),
    "balance_gbp":        round(float(clean["Amount"].sum()), 2),
    "sku_period_rows":    int(len(sku_period)),
    "sku_month_rows":     int(len(sku_month)),
    "rfm_rows":           int(len(rfm_export)),
    "sales_with_customer":    int((~clean["IsCancel"] & clean["Customer ID"].notna()).sum()),
    "sales_without_customer": int((~clean["IsCancel"] & clean["Customer ID"].isna()).sum()),
}

path = PROCESSED / "etl_summary.json"
path.write_text(json.dumps(summary, ensure_ascii=False, indent=2))

print("ИТОГ ETL")
for k, v in summary.items():
    print(f"  {k:<24} {v}")

ИТОГ ETL
  raw_rows                 1067371
  clean_rows               996172
  sales_rows               978604
  return_rows              17568
  revenue_sales_gbp        19029370.72
  returns_gbp              542371.21
  balance_gbp              18486999.51
  sku_period_rows          8126
  sku_month_rows           97512
  rfm_rows                 8532
  sales_with_customer      759632
  sales_without_customer   218972


## Что получилось

В папке `data/processed/`:

| Файл | Гранулярность | Для чего в BI |
|---|---|---|
| `DimProduct.parquet` | 1 SKU | Справочник товаров |
| `DimDate.parquet` | 1 день | Календарь для динамики |
| `DimCustomer.parquet` | 1 покупатель | Справочник клиентов |
| `DimPeriod.parquet` | 2 периода | Границы годовых окон |
| `FactSales.parquet` | 1 позиция чека | Страница «Обзор бизнеса» |
| `MartSkuPeriod.parquet` | SKU × период | Страница «Конструктор сценариев» |
| `MartSkuMonth.parquet` | SKU × месяц | Графики по месяцам |
| `MartRfm.parquet` | покупатель × период | RFM-страница |
| `etl_summary.json` | — | Журнал прогона |

**Что НЕ выгружаем:**
- `candidates_soft`, `candidates_moderate`, `candidates_aggressive` — сценарии считаются в DAX.
- Флаги `scenario_*` — заменяются мерами Power BI.

